# N-gram Sentence Prediction

This notebook builds a very simple sentence generator from the quote dataset.

It uses:
- 4-gram if the context is available
- trigram if 4-gram is not available
- bigram if trigram is not available
- unigram as the final fallback

It also handles unknown words with an `<unk>` token.

# Setup and Imports

Load the libraries needed for tokenization, preprocessing, and model training.

In [10]:
import math
import re
from collections import Counter, defaultdict
import nltk
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download required tokenization dataset
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
print("Setup completed successfully.")

Setup completed successfully.


# Load the Dataset

Read the CSV file and build the training corpus used by the language model.

In [12]:
# Load the dataset from CSV
dataset_path = "qoute_dataset.csv"

try:
    df = pd.read_csv(dataset_path)
    # Combine text from all rows (handles string columns automatically)
    corpus_text = " ".join(df.astype(str).values.flatten())
    print(
        f"Loaded '{dataset_path}' successfully with {len(corpus_text)} characters."
    )
except Exception as e:
    print(f"Error reading dataset: {e}")


def preprocess_corpus(corpus_text, min_freq=2):
    """Tokenizes corpus text, lowers case, and replaces low-frequency words with <unk>."""
    # Tokenize full corpus
    tokens = word_tokenize(corpus_text.lower())

    # Count term frequencies
    freqs = Counter(tokens)

    # Replace rare words (frequency < min_freq) with <unk>
    processed_tokens = []
    vocab = set()

    for token in tokens:
        if freqs[token] >= min_freq:
            processed_tokens.append(token)
            vocab.add(token)
        else:
            processed_tokens.append("<unk>")

    vocab.add("<unk>")
    return processed_tokens, vocab


def process_user_input(input_text, vocab):
    """Converts user input tokens to <unk> if they are missing from vocabulary."""
    tokens = word_tokenize(input_text.lower())
    return [token if token in vocab else "<unk>" for token in tokens]

Loaded 'qoute_dataset.csv' successfully with 520966 characters.


# Preprocess the Corpus

Tokenize the dataset, normalize words, and replace rare tokens with `<unk>`.

In [13]:
class LaplaceBigramModel:

    def __init__(self, corpus_tokens, vocab):
        self.vocab = list(vocab)
        self.vocab_set = vocab
        self.vocab_size = len(vocab)
        self.unigram_counts = Counter(corpus_tokens)
        self.bigram_counts = defaultdict(Counter)

        # Count bigram pairs across the processed dataset
        for w1, w2 in zip(corpus_tokens[:-1], corpus_tokens[1:]):
            self.bigram_counts[w1][w2] += 1

    def get_bigram_prob(self, w1, w2):
        """Calculates P(w2 | w1) using Laplace (+1) Smoothing."""
        count_w1_w2 = self.bigram_counts[w1][w2]
        count_w1 = self.unigram_counts[w1]

        # Laplace smoothing formula
        return (count_w1_w2 + 1) / (count_w1 + self.vocab_size)

    def predict_next_word(self, current_word):
        """Probabilistically samples next word from vocabulary based on Laplace probabilities."""
        # Calculate transition probabilities for all items in vocabulary
        probs = np.array(
            [self.get_bigram_prob(current_word, w) for w in self.vocab]
        )

        # Normalize probabilities so sum equals 1.0
        probs = probs / np.sum(probs)

        # Sample next word probabilistically
        return np.random.choice(self.vocab, p=probs)

    def generate_sentence(self, prompt_text, max_words=15):
        """Generates sequence starting from user input prompt."""
        processed_prompt = process_user_input(prompt_text, self.vocab_set)

        if not processed_prompt:
            sentence = ["<unk>"]
            current_word = "<unk>"
        else:
            sentence = processed_prompt.copy()
            current_word = processed_prompt[-1]

        for _ in range(max_words):
            next_word = self.predict_next_word(current_word)

            # Prevent repeating consecutive <unk> tokens
            if next_word == "<unk>" and sentence[-1] == "<unk>":
                continue

            sentence.append(next_word)
            current_word = next_word

            # Break output on standard punctuation marks
            if next_word in [".", "!", "?"]:
                break

        return " ".join(sentence)

# Train the N-Gram Model

Build the Laplace-smoothed bigram model from the processed corpus.

In [14]:
# Preprocess training corpus loaded from quote_dataset.csv
# Set min_freq=2 to ensure rare dataset words are converted to <unk> for smoothing training
corpus_tokens, vocab = preprocess_corpus(corpus_text, min_freq=2)

# Instantiate and train model
model = LaplaceBigramModel(corpus_tokens, vocab)

print("--- Model Training Complete ---")
print(f"Total Tokens Processed : {len(corpus_tokens)}")
print(f"Vocabulary Size (|V|)  : {len(vocab)}")
print("Sample Vocabulary      :", list(vocab)[:10])

--- Model Training Complete ---
Total Tokens Processed : 114661
Vocabulary Size (|V|)  : 4318
Sample Vocabulary      : ['dangerously', 'badass', 'names', 'twenty', 'terrors', 'wander', 'kills', 'meanings', 'created', 'just']


# Generate a Sentence

Enter a prompt and let the model generate the next words.

In [15]:
# Interactive sentence generation cell
user_prompt = input("Enter a starting phrase: ")

if user_prompt.strip():
    generated_text = model.generate_sentence(user_prompt, max_words=15)
    print("\n--- Generation Output ---")
    print(f"User Input : {user_prompt}")
    print(f"Generated  : {generated_text}")
else:
    print("Please enter a non-empty phrase.")


--- Generation Output ---
User Input : If
Generated  : if augustus hard preserved bat idea gap scarce frightens tearing weary city entreating loves individual elie
